# Part 2: the same loop on 20,663 AlphaEarth embeddings of Austria, 1.25% solar farms

The same loop as in part 1, on real data: 20,663 locations in Austria, each described by its AlphaEarth embedding, with 259 solar farms among them (1.25%). This is the dataset behind the simulation figures of the EarthQuery report.

An AlphaEarth embedding is a list of 64 numbers that Google computes for every 10 m pixel on Earth from a year of satellite images. Two pixels with similar embeddings look similar from space. We never look at the images here: the model works on the 64 numbers only.

This notebook shows the probability map after each round of one run, which of the three criteria of the EarthQuery rule found which farms, and the six strategies compared over 50 rounds.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))      # makes `al_training` importable from this folder
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
os.makedirs("../outputs", exist_ok=True)

In [ ]:
d = np.load("../data/austria_pool.npz")
lon, lat, y, X = d["lon"], d["lat"], d["label"].astype(int), d["emb"]
print(f"{len(y):,} locations, {y.sum()} solar farms ({y.mean():.2%}), {X.shape[1]} numbers each")

In [ ]:
from al_training.plotting import draw_austria

fig, ax = plt.subplots(figsize=(9, 5))
k = draw_austria(ax)                                     # k shrinks longitudes so the map is not stretched
ax.scatter(lon * k, lat, s=2, color="#b8b8b8", lw=0, label="other locations")
ax.scatter(lon[y == 1] * k, lat[y == 1], s=14, color="#e08a1e", lw=0, label="solar farms")
ax.legend(loc="lower left", frameon=False); ax.set_title(f"the pool: {len(y):,} locations, {y.sum()} solar farms", fontsize=10);

## The input: the 64 embedding values of a solar farm and of an ordinary location

Here are the 64 numbers of a typical solar farm and of a typical ordinary location (each the one closest to its class average), and below them the average farm minus the average other location: the direction of solar farms in embedding space. We cannot read the numbers, but the model can compare them. The colour of each bar says how much the Austrian solar farm model relies on that dimension, measured as the mean absolute SHAP value of the random forest in the EarthQuery three-country study; the three most important dimensions are named. They are also the ones that carry the largest difference between farms and the rest, and part 4 shows them as red, green and blue over the country.

In [ ]:
from al_training.plotting import plot_embedding_bars

fig = plot_embedding_bars(X, y)                          # colour = importance for the model (mean |SHAP|)

## Pool and test set: 90% and 10% of the locations

We keep 10% of the locations aside as a test set and use the rest as the pool. In the pool we pretend the labels are unknown: a strategy chooses locations, and the "oracle" (the true label, in a real campaign a person) reveals them.

In [ ]:
from al_training.simulate import split_pool

pool, test = split_pool(X, y, test_fraction=0.1, seed=0)
print(f"pool: {pool.sum():,} locations with {y[pool].sum()} farms;  test: {test.sum():,} locations with {y[test].sum()} farms")

## Animation: one run of the EarthQuery rule, 50 rounds of 6 labels, the probability map after each round

The model is the one used for the report's figures: standardise the 64 numbers, drop 75% of them at random during training (a strong regulariser), one linear layer. It trains in milliseconds, so 50 rounds take seconds. The EarthQuery tool itself uses a random forest inside Earth Engine; pass `model="rf"` to try it here (slower).

We run the EarthQuery rule for 50 rounds of 6 labels (2 per criterion) and record the model's probability for every pool location after each round.

In [ ]:
from al_training.simulate import run_active_learning

h = run_active_learning(X[pool], y[pool], X[test], y[test], strategy="earthquery",
                        n_iter=50, batch=6, model="probe", seed=0, record_probs=True)
print(f"{h['n_labels'][-1]} labels, {h['n_found'][-1]} of {y[pool].sum()} farms labelled, "
      f"balanced accuracy on the test set {h['bal_acc'][-1]:.2f}, farm recall {h['recall_target'][-1]:.2f}")

The map shows the model's current probability of a solar farm over Austria in the colours of the Earth Engine layer of part 4: dark = low, yellow = high. The pool is a sample of locations about 2 km apart, not every pixel, so the map is drawn on a 1 km grid in which each cell shows the highest probability among the pool locations within 2.5 km, which is also what the coarse search of the tool looks at (the peak of each cell). Labelled locations appear as they are chosen: orange for farms, white for the rest, the newest batch ringed. The right panel counts the farms found, in total and by criterion, against what random sampling would find, and the title counts the negatives labelled.

In [ ]:
from IPython.display import HTML
from al_training.plotting import AustriaFrames, save_gif

frames = AustriaFrames(h, lon[pool], lat[pool], y[pool])
anim = frames.animation(interval=200)
save_gif(anim, "../outputs/austria_earthquery.gif", fps=5)
HTML(anim.to_jshtml())

## The probability map after rounds 1, 3, 5, 20 and 50

The same run after rounds 1, 3, 5, 20 and 50. After one round the model has one farm and seven negatives and gives most of the country a high probability: with so few labels it cannot separate farms from the rest. After three rounds most of Austria has a low probability and a few hot spots remain. After five rounds the hot spots are in the east, where most Austrian solar farms are. By round 20 the map has its final shape; the remaining rounds sharpen it and label the farms.

In [ ]:
from al_training.plotting import probability_snapshots

fig = probability_snapshots(h, lon[pool], lat[pool], y[pool], rounds=(1, 3, 5, 20, 50), figsize=(20, 3.6))
fig.savefig("../outputs/austria_probability_rounds.png", dpi=130, bbox_inches="tight")

The same run with a slider, if you prefer to step through the rounds yourself. The slider gets its own copy of the figure; sharing one figure between an animation and a slider makes the notebook slow.

In [ ]:
from ipywidgets import interact, IntSlider

slider = AustriaFrames(h, lon[pool], lat[pool], y[pool])
interact(slider.show_round, i=IntSlider(value=0, min=0, max=50, description="round"));

## Which criterion chose each location, and how many farms each criterion found

Every pick records the criterion that chose it. Here is one round, drawn on the probability map of that round, followed by the count over the whole run.

In [ ]:
from al_training.plotting import plot_round_picks

fig, ax = plt.subplots(figsize=(10, 6))
plot_round_picks(ax, h, lon[pool], lat[pool], y[pool], i=12)

In [ ]:
from al_training.simulate import picks_by_source
from al_training.plotting import plot_strata_bars

table = picks_by_source(h, y[pool])
fig, ax = plt.subplots(figsize=(6.5, 4))
plot_strata_bars(ax, table, title="picks and farms found per criterion, 50 rounds of 6 labels")
table

Most exploit and diversity picks are farms. Novelty finds fewer farms by design: it picks high-scoring locations unlike any labelled one, so it returns the farms of a new type and the hard negatives that the other two criteria do not pick. The global campaign shows the same pattern: exploit has the highest yield in every region, and novelty returned 0 farms in 300 candidates in a sparse region such as Canada 8 but 197 in 300 in the dense United States 18.

## All six strategies: farms found, balanced accuracy and background recall

Now every strategy from part 1 plus the EarthQuery rule, each run twice from different starting pairs (the report used three starts and 2 labels per round; this takes about a minute). Three scores: how many farms were labelled, balanced accuracy on the test set, and the recall of the background, which shows whether the model still recognises ordinary land.

In [ ]:
from al_training.simulate import run_many
from al_training.strategies import NAMES

results = run_many(X[pool], y[pool], X[test], y[test], NAMES, seeds=(0, 1), n_iter=50, batch=6, model="probe")

In [ ]:
from al_training.plotting import plot_learning_curves

fig, axes = plt.subplots(1, 3, figsize=(17, 4.2))
plot_learning_curves(results, "n_found", ax=axes[0], title=f"solar farms labelled (of {y[pool].sum()})")
plot_learning_curves(results, "bal_acc", ax=axes[1], chance=0.5, title="balanced accuracy", legend=False)
plot_learning_curves(results, "recall_other", ax=axes[2], title="recall of the background", legend=False)

In [ ]:
import pandas as pd

rows = []
for name, runs in results.items():
    rows.append({"strategy": name,
                 "farms labelled": np.mean([r["n_found"][-1] for r in runs]),
                 "balanced accuracy": np.mean([np.mean(r["bal_acc"][-10:]) for r in runs]),
                 "farm recall": np.mean([np.mean(r["recall_target"][-10:]) for r in runs]),
                 "background recall": np.mean([np.mean(r["recall_other"][-10:]) for r in runs])})
pd.DataFrame(rows).set_index("strategy").round(2)     # scores: mean of the last 10 rounds (about 300 labels)

What to notice:

* Random and TypiClust fall to chance. With 1.25% farms, random sampling labels about 4 farms in 300, and TypiClust covers the landscape types, not the rare class. Both start with one farm, and after a few rounds of negatives the model predicts no farms at all. Covering the data is not enough when the class we want is rare.
* Entropy finds farms, but slowly, because it only asks near the boundary it already has.
* Exploit only labels the most farms and reaches the best balanced accuracy. Its background recall dips a little in the second half, when the easy farms are used up and it starts collecting hard negatives; in the report, with 2 labels per round, that dip goes down to 0.47.
* Exploit + entropy and the EarthQuery rule find almost as many farms while keeping the background recall at 1.

In the report (2 labels per round, three starts, 300 labels) the numbers are: exploit only 0.91 balanced accuracy, exploit + entropy 0.91, entropy 0.78 to 0.81, random and TypiClust 0.50.

## Animation: random sampling on the same pool, 50 rounds of 6 labels

For comparison, the same animation for random sampling: the probability map stays low everywhere, because the model almost never receives a farm label.

In [ ]:
h_random = run_active_learning(X[pool], y[pool], X[test], y[test], strategy="random",
                               n_iter=50, batch=6, model="probe", seed=0, record_probs=True)
frames_random = AustriaFrames(h_random, lon[pool], lat[pool], y[pool])
anim_random = frames_random.animation(interval=200)
save_gif(anim_random, "../outputs/austria_random.gif", fps=5)
HTML(anim_random.to_jshtml())

## Summary

* The loop from part 1 works unchanged on 64-dimensional embeddings, and the probability map is the model's prediction after each round.
* At 1% prevalence, the strategies that go looking for the target class (exploit, exploit + entropy, the EarthQuery rule) are the only ones whose model improves; random and TypiClust never find it.
* In the EarthQuery rule, exploit and diversity bring the farms, novelty brings the farms of a new kind and the hard negatives, and together they keep the model good on the background while it collects farms.

Part 3 shows what this loop produced at global scale, and part 4 lets you run it live on the map.